In [1]:
import torchvision as thv

from common.nn.dataset.nn_dataset import NNDataset

from common.nn.enum.nets import Nets
from common.nn.enum.losses import Losses
from common.nn.enum.devices import Devices

from common.nn.params.nn_run import NNRun
from common.nn.params.nn_params import NNParams
from common.nn.params.nn_train_params import NNTrainParams
from common.nn.params.nn_model_params import NNModelParams

from common.nn.nn_model import NNModel

from common.utils import Utils
from common.vis_utils import VisUtils

In [2]:
DS_MEAN : float = 0.1307
DS_STD  : float = 0.3081

ds = NNDataset(
    ds_class=thv.datasets.MNIST
    , transform=thv.transforms.Compose(
        [
            thv.transforms.ToTensor()
            , thv.transforms.Normalize(mean=DS_MEAN, std=DS_STD)
        ]
    )
)

print(
    ds.input_dim
    , ds.output_dim
    , len(ds.train_loader.dataset)
    , len(ds.val_loader.dataset)
    , len(ds.test_loader.dataset)
)

784 10 60000 1000 9000


In [3]:
n_epochs = 100
optims = ["adam"]

dropout_probs = [0.5]
hidden_dimss = [
    []
    # , [500]
    # , [1000]
    # , [1000, 500]
    # , [1000, 500, 250]
    # [1000, 500, 250, 125]
]

models = [
    model
        for model in [
            NNModel(
                params=NNModelParams(
                    net=Nets.FEED_FWD
                    , device=Devices.CPU
                    , loss=Losses.CROSS_ENTROPY
                )
                , net_params=NNParams(
                    dropout_prob=dropout_prob
                    , hidden_dims=hidden_dims
                    , input_dim=ds.input_dim
                    , output_dim=ds.output_dim
                )
            )
                for dropout_prob in dropout_probs
                for hidden_dims in hidden_dimss
        ]
]

train_params = [
    train_param
        for train_param in [
            NNTrainParams(n_epochs=n_epochs)
                .with_train_loader(value=ds.train_loader)
                .with_val_loader(value=ds.val_loader)
        ]
]

runs = [
    run for run in [
        model.train(params=train_param)
            for model in models
            for train_param in train_params
    ]
]

+--------------------------------------------------------------+
|                        Run Details...                        |
+---------------------------+----------------------------------+
|             id            | b847c22d7f8433b2f480c66c84ccedab |
|         model.net         |             feed_fwd             |
|         model.loss        |          cross_entropy           |
|        model.device       |               cpu                |
|       net.input_dim       |               784                |
|       net.output_dim      |                10                |
|      net.dropout_prob     |               0.5                |
|      net.hidden_dims      |                []                |
|       net.activation      |            leaky_relu            |
|       train.n_epochs      |               100                |
|     train.optim.max_lr    |               0.9                |
|    train.optim.momentum   |           (0.9, 0.999)           |
|      train.optim.name  

Training: 100%|██████████| 100/100 [04:16<00:00,  2.57s/it, error=0.0990, lr=0.2288]

In [2]:
top_runs = [
    run for run in sorted(
        NNRun.all()
        , key=lambda run: min(
            run.idps, key=lambda idp: idp.val_edp.error
        ).val_edp.error
    )[:3]
]

In [3]:
VisUtils.multi_line_plot(
    x_ticks_inc=5
    , y_axis_label="LR"
    , x_axis_label="Iterations"
    , title=f"Learning Rates"
    , x=[
        iter_idx for iter_idx in range(
            0
            , max(
                top_runs
                , key=lambda run: run.idps[-1].iter_idx
            ).idps[-1].iter_idx
        )
    ]
    , yss_legend=[
        [
            f"lr of {run.id}"
        ] for run in top_runs
    ]
    , yss=[
        [
            [idp.lr for idp in run.idps]
        ] for run in top_runs
    ]
)

In [4]:
VisUtils.multi_line_plot(
    x_ticks_inc=5
    , y_axis_label="Loss"
    , x_axis_label="Iterations"
    , title=f"Training & Validation Losses"
    , x=[
        iter_idx for iter_idx in range(
            0
            , max(
                top_runs
                , key=lambda run: run.idps[-1].iter_idx
            ).idps[-1].iter_idx
        )
    ]
    , yss_legend=[
        [
            f"{loss_type} of {run.id}" for loss_type in ["training", "validation"]
        ] for run in top_runs
    ]
    , yss=[
        [
            [idp.train_edp.loss for idp in run.idps]
            , [idp.val_edp.loss for idp in run.idps]
        ] for run in top_runs
    ]
)

In [18]:
VisUtils.multi_line_plot(
    x_ticks_inc=5
    , y_axis_label="Error"
    , x_axis_label="Iterations"
    , title=f"Training & Validation Errors"
    , x=[
        iter_idx for iter_idx in range(
            0
            , max(
                top_runs
                , key=lambda run: run.idps[-1].iter_idx
            ).idps[-1].iter_idx
        )
    ]
    , yss_legend=[
        [
            f"{error_type} of {run.id}" for error_type in ["training", "validation"]
        ] for run in top_runs
    ]
    , yss=[
        [
            [idp.train_edp.error for idp in run.idps]
            , [idp.val_edp.error for idp in run.idps]
        ] for run in top_runs
    ]
)

In [7]:
print(f"best run is {top_runs[0].id} which achieves validation error of {min(top_runs[0].idps, key=lambda idp: idp.val_edp.error).val_edp.error:.4f}")

best run is 1da1484e7a5f96dd4564b119d21bfc7c which achieves validation error of 0.0870


In [8]:
N_SAMPLES = 5_000

for checkpoint in NNRun.load("best").checkpoints():
    VisUtils.two_dim_tsne_checkpoint_logits(checkpoint=checkpoint, ds=ds, n_samples=N_SAMPLES)